In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def build_actor(state_dim=5, hidden=256, n_layers=3):
    """
    Outputs:
        actor: nn.Module
    Forward:
        state: (B, state_dim) -> action: (B, 1) in [0,1]
    """
    layers = []
    in_dim = state_dim

    for _ in range(n_layers):
        layers.append(nn.Linear(in_dim, hidden))
        layers.append(nn.ReLU())
        in_dim = hidden

    layers.append(nn.Linear(in_dim, 1))
    layers.append(nn.Sigmoid())  # bornage [0,1]

    actor = nn.Sequential(*layers)
    return actor

In [ ]:
class CriticQuantile(nn.Module):
    def __init__(self, state_dim=5, action_dim=1, M=100, h1=512, h2=512, h3=256):
        super().__init__()
        self.M = M

        in_dim = state_dim + action_dim
        self.fc1 = nn.Linear(in_dim, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, h3)
        self.out = nn.Linear(h3, M)

    def forward(self, state, action):
        # concatenate state and action and applies 3 layers ReLu and outputs the quantiles vector
        """
        state: (B, state_dim)
        action: (B, action_dim) typically (B,1)
        returns quantiles: (B, M)
        """
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        q = self.out(x)
        return q


def build_critic_quantile(state_dim=5, action_dim=1, M=100, h1=512, h2=512, h3=256):
    """
    Outputs:
        critic: nn.Module
    Forward:
        (state, action) -> quantiles: (B, M)
    """
    return CriticQuantile(state_dim=state_dim, action_dim=action_dim, M=M, h1=h1, h2=h2, h3=h3)

In [ ]:
def actor_forward(actor, state):
    # Aapplies 3 layers ReLu and outputs the action in [0,1]
    """
    Inputs:
        state: torch.Tensor (B, 5)
    Outputs:
        action: torch.Tensor (B, 1) in [0,1]
    """
    return actor(state)

In [5]:
def critic_forward(critic, state, action):
    """
    Outputs:
        quantiles: torch.Tensor (B, M)
    """
    return critic(state, action)

In [ ]:
RUN_TEST = False

if RUN_TEST:
    torch.manual_seed(0)

    B = 8  #Batch size
    state_dim = 5
    action_dim = 1
    M = 100

    actor = build_actor(state_dim=state_dim, hidden=256, n_layers=3)
    critic = build_critic_quantile(state_dim=state_dim, action_dim=action_dim, M=M)

    state = torch.randn(B, state_dim)

    action = actor_forward(actor, state)
    quantiles = critic_forward(critic, state, action)

    print("action shape:", action.shape)
    print("action min/max:", float(action.min()), float(action.max()))
    print("quantiles shape:", quantiles.shape)

action shape: torch.Size([8, 1])
action min/max: 0.4956408143043518 0.511164665222168
quantiles shape: torch.Size([8, 100])


C:\Users\houss\AppData\Local\Temp\ipykernel_13480\3777069886.py:20: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print("action min/max:", float(action.min()), float(action.max()))
